# Functional programming in Python
## A smart-home story (continuation from the lecture and first class): from a useful script to a dependable data pipeline

**Student notebook - You will find here some guided examples, plenty of explanations, and 15 exercises**

The initial toy project has told you a story about becoming a pro by expanding beyond of "a single script with a single purpose". A temperature sensor reports a few numbers,
and a script tells us the average temperature and how many readings are at least some degrees of Celsius.
Now another room needs a different threshold. A replacement sensor reports Fahrenheit.
Someone wants to keep the original measurements for an analysis. Eventually, readings will
arrive one at a time (when a sensor is triggered) rather than as one convenient list.

We will let those requirements lead us to **functional programming**. We will not start with
a finished package or a collection of unfamiliar abstractions. Each new idea will appear when we propose a solution a problem we encounter.

### Before we start - prerequisites

We need to know/learn:

- variables,
- `if`,
- `for`,
- lists,
- dictionaries,
- and how to define a simple function.

No previous knowledge of functional programming, classes, decorators, or
generators is assumed. New terms are introduced when they become useful. If you want to run it on your own system - you need python at least **3.10**.

We will run the code in Jupyter on Google Colab. All computation and exercise
checks use the standard library. Two demonstration plots use Matplotlib; their cell
explains how to install it when it is missing (should not be necessary here but it is good to know for the future).

The exercise functions initially raise `NotImplementedError`. A small, visible checking helper reports these as
**TODO**, so an untouched notebook can run to the end as is, but will need some work to remove all place holders by adding code. Replace the placeholder body,
then rerun that exercise and its check. Actual wrong answers are not hidden. We will be working with the notebooks during the lab classes.

### Summary

| Part | What changes in the project? | Main ideas (to be follow up during homework) |
|:--|:--|:--|
| [1–2](#s01) | Results must be repeatable; raw data must survive. | Pure functions, side effects, mutation, copying. |
| [3–5](#s03) | Rooms need different calculations and selection rules. | Functions as values, higher-order functions, `lambda`, `map`, `filter`, comprehensions. |
| [6–8](#s06) | Policies and processing steps need to be reusable. | Closures, `partial`, composition, reductions. |
| [9–10](#s09) | Measurements arrive as a stream. | Iterators, generators, laziness, `itertools`. |
| [11–13](#s11) | Bad input, device state, and reporting must be handled clearly. | Functional core, explicit state, tests, a small capstone. |
| [14–15](#s14) | Optional extensions. | Recursion on nested data, memoization. |
| [16](#s16) | Choose a style rather than follow a rule. | Summary, glossary, and further reading. |


In [288]:
import sys
from collections.abc import Callable
from copy import deepcopy
from functools import partial, reduce, lru_cache
from itertools import accumulate, chain, count, islice, pairwise, groupby
from math import isclose, isfinite, sin, cos, pi
from statistics import mean
from typing import Any

print("Python:", sys.version.split()[0])

Python: 3.14.5


### How the exercise checks work

`assert condition` checks that a condition is true. A false condition raises an
`AssertionError`, which points us towards a mistake. These exercises checks are just examples of expected behaviour, not a proof that every possible input works.

**Note**, Python can disable assertions in optimized execution; do not use them as your application's input validation.

The first helper below only makes unfinished exercises convenient to run. The second checks that an invalid input raises the **expected type** of exception. Neither helper implements any part of the sensor project - they can be used as convenient code to share between projects.

In [289]:
exercise_status = {}


def run_check(label, check):
    exercise_status[label] = "running"
    try:
        check()
    except NotImplementedError:
        exercise_status[label] = "TODO"
        print(f"{label}: TODO — replace the placeholder, then rerun this check.")
    except Exception:
        exercise_status[label] = "failed"
        raise
    else:
        exercise_status[label] = "passed"
        print(f"{label}: all supplied checks passed.")


def assert_raises(exception_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except exception_type:
        return
    raise AssertionError(f"Expected {exception_type.__name__}")

<a id="s01"></a>
# 1. From a script to a computing engine

> **New requirement:** “The dashboard and the email report must calculate the same answer - add an universal part, no repeated code.”

Here is our starting point. There is nothing wrong with this small script: its inputs and its actions fit on one screen. The difficulty appears when other parts of the project need the calculation without this exact printing behaviour.

In [290]:
temperatures_c = [21.0, 22.0, 24.0]
threshold_c = 23.0
average_c = sum(temperatures_c) / len(temperatures_c)
hot_count = sum(value >= threshold_c for value in temperatures_c)
print(f"Average: {average_c:.2f} °C")
print(f"Readings at or above {threshold_c:.1f} °C: {hot_count}")

Average: 22.33 °C
Readings at or above 23.0 °C: 1


Read `sum(value >= threshold_c for value in temperatures_c)` as “count the comparisons that are true.” In counting, `True` contributes 1 and `False` contributes 0. This is a bit more compact form later; an ordinary counting loop would be equally legitimate - no problem.

Let us put the calculation in a function and leave printing outside it. Our input contract here is deliberately simple: a **non-empty list or tuple of finite number of temperatures in Celsius**, plus a finite threshold. Validation of raw sensor messages comes later.

In [291]:
def summarise(values_c, threshold_c=23.0):
    """Summarise a non-empty list or tuple; do not modify it."""
    if not values_c:
        raise ValueError("At least one temperature is required.")
    return {
        "count": len(values_c),
        "mean_c": sum(values_c) / len(values_c),
        "hot_count": sum(value >= threshold_c for value in values_c),
    }


summary = summarise(temperatures_c, threshold_c=23.0)
""" Displaying the result is a separate action now """
print(summary)

{'count': 3, 'mean_c': 22.333333333333332, 'hot_count': 1}


A **pure function**, in the sense used here, has two important properties:
its result is determined by its explicit inputs, and calling it does not change anything outside the function. For equal valid inputs, `summarise` returns equal results. It does not edit the supplied list, change a global setting, read the current clock, or send a message. It returns a fresh dictionary; purity does not mean that this returned Python dictionary is itself immutable.

A **side effect** is an observable action beyond returning the calculation's result:

- printing,
- writing a file,
- modifying a supplied dictionary,
- etc.

Reading external state is also a dependency to watch: a function that reads a sensor or a clock cannot generally be treated as a calculation determined solely by its arguments.

The distinction is not somehow “good functions versus bad functions.” A real temperature dashboard must show something - that is a common sense. We want to put the calculation and the action in places where each action is clear. This is a practical functional style in a language that also supports other styles.

See the [official Functional Programming HOWTO](https://docs.python.org/3.13/howto/functional.html#introduction).

### A hidden input is still an input

In [292]:
current_threshold_c = 23.0


def is_hot_hidden(value_c):
    return value_c >= current_threshold_c


print(is_hot_hidden(24.0))
current_threshold_c = 25.0
print(is_hot_hidden(24.0))  # Same argument, different answer.


def is_hot(value_c, threshold_c):
    return value_c >= threshold_c


print(is_hot(24.0, 23.0), is_hot(24.0, 25.0))

True
False
True False


The first function does not mutate anything, but its answer still depends on a changing name outside its argument list. “Does not modify its input” is therefore not a complete definition of purity. In the second version, the policy is visible at the call site.

For an ordinary value calculation, **referential transparency** means that we can replace a call by its result without changing the program's observable meaning. Replacing `is_hot(24.0, 23.0)` by `True` is harmless here. Replacing a call that prints a message by its returned value would remove that message. We will use this idea informally for value-based results, not to make promises about Python object identity or execution time.

**Predict before running:** should asking for the same summary twice change the input list? Should changing the threshold change the average? These are useful questions to turn into tests.

In [293]:
before = temperatures_c.copy()
assert summarise(temperatures_c, 23.0) == summarise(temperatures_c, 23.0)
assert temperatures_c == before
assert (
    summarise(temperatures_c, 23.0)["mean_c"]
    == summarise(temperatures_c, 25.0)["mean_c"]
)
print("Repeated calculations agree; the input is unchanged.")

Repeated calculations agree; the input is unchanged.


<a id="e01"></a>

## Ex1: Calculate a heating gap

The controller first needs a number, not an action. Write
`heating_gap(temperature_c, target_c)` to return how far the room is below its target. Return `0.0` when the room is already at or above the target. Assume finite numerical inputs. Do not print, read a global target, or change any outside object.

For example, a room at 18.5 °C with a 21.0 °C target has a gap of 2.5 °C.

Add a text cell with your comments - especially, interpret the tests implemented in check_ex1 function.

**Hint.** Calculate target minus measurement, then decide what a negative result should mean.

In [294]:
def heating_gap(temperature_c: float, target_c: float) -> float:
    return max(0.0, target_c - temperature_c)

In [295]:
def check_ex1():
    assert isclose(heating_gap(18.5, 21.0), 2.5)
    assert heating_gap(21.0, 21.0) == 0.0
    assert heating_gap(24.0, 21.0) == 0.0
    assert isclose(heating_gap(-2.0, 1.0), 3.0)
    assert heating_gap(18.5, 21.0) == heating_gap(18.5, 21.0)


run_check("Ex1", check_ex1)

Ex1: all supplied checks passed.


<a id="s02"></a>

# 2. Keep the original measurements

> **New requirement:** “We need the raw measurements for any future analysis and additional checks.”

Say, the calibration procedure adds an offset. An easy implementation edits the list it receives. That becomes a problem when another part of the program still interprets that this list contains the raw measurements.

In [296]:
def calibrate_in_place(values_c, offset_c):
    for index in range(len(values_c)):
        values_c[index] += offset_c
    return values_c


raw_example = [20.0, 21.0]
""" This an alias, not another list """
audit_reference = raw_example
calibrated_values = calibrate_in_place(raw_example, 0.5)
print("Raw:", raw_example)
print("Audit reference:", audit_reference)
print("Same object:", raw_example is audit_reference)
print("Calibrated:", calibrated_values)

Raw: [20.5, 21.5]
Audit reference: [20.5, 21.5]
Same object: True
Calibrated: [20.5, 21.5]


**Mutation** means changing an existing object. Here both names refer to the same list, so both see the changes. Assignment did not make a copy.

We can instead make a new result. Notice the loop and `append` below. A functional interface does not require banning loops: this list is created inside the function and is not shared with the caller while it is being built.

In [297]:
def calibrated_values(values_c, offset_c):
    result = []
    for value_c in values_c:
        result.append(value_c + offset_c)
    return result


raw_example = [20.0, 21.0]
calibrated_values = calibrated_values(raw_example, 0.5)
print("Raw:", raw_example)
print("Corrected:", calibrated_values)
assert raw_example == [20.0, 21.0]

Raw: [20.0, 21.0]
Corrected: [20.5, 21.5]


An **immutable object** cannot have its own contents changed after creation. Numbers, strings, and tuples are useful examples. Lists and dictionaries are mutable. In Python we often work functionally with ordinary lists and dictionaries by agreeing not to edit input objects and by returning new results when something changes.

This agreement is not the same as enforcement. A tuple cannot have an element reassigned, but a mutable object stored inside it can still change. Likewise, a new dictionary made with `.copy()` shares its nested values with the original. This is a **shallow copy**.

The [copy documentation](https://docs.python.org/3.13/library/copy.html) explains the shallow/deep distinction; the [built-in types reference](https://docs.python.org/3.13/library/stdtypes.html#typesseq) describes mutable and immutable sequences.

In [298]:
original = {"sensor_id": "kitchen", "labels": ["indoor"]}
shallow = original.copy()
shallow["labels"].append("checked")
print("Original after editing the shallow copy:", original)

nested_tuple = ([20.0, 21.0], "kitchen")
nested_tuple[0].append(22.0)
print("The tuple still contains a mutable list:", nested_tuple)

Original after editing the shallow copy: {'sensor_id': 'kitchen', 'labels': ['indoor', 'checked']}
The tuple still contains a mutable list: ([20.0, 21.0, 22.0], 'kitchen')


The lesson is not “deep-copy everything.” Copy only what needs to be independent, and make that ownership decision explicit. In our small record, the sensor ID is a string and needs no copying. The list of labels does need a new list when we add a label.
For very large data, repeated copying may be expensive; avoiding mutation is a design choice to weigh against memory and time.

<a id="e02"></a>

## Ex2: Add a label without rewriting history

Write `add_label(record, label)` to return a new dictionary with `label` appended to a new `labels` list. Preserve the other fields. If `labels` is absent, treat it as an empty list. Existing labels are strings; adding the same label twice is allowed.

Neither the input dictionary nor its existing labels list may change. The result's labels list must be independent of the original list. Other fields may be shallowly shared because this function does not change them.

**Hint.** `record.get("labels", [])` supplies a default for a missing key. Concatenating a list with `[label]` creates a new list.

In [299]:
def add_label(record: dict[str, Any], label: str) -> dict[str, Any]:
    return {
        **record,
        "labels": record.get("labels", []) + [label],
    }

In [300]:
def check_ex2():
    original = {"sensor_id": "kitchen", "labels": ["indoor"], "value": 21.0}
    snapshot = deepcopy(original)
    result = add_label(original, "checked")
    assert result == {
        "sensor_id": "kitchen",
        "labels": ["indoor", "checked"],
        "value": 21.0,
    }
    assert original == snapshot
    assert result is not original
    assert result["labels"] is not original["labels"]
    result["labels"].append("later")
    assert original == snapshot
    assert add_label({"sensor_id": "study"}, "new") == {
        "sensor_id": "study",
        "labels": ["new"],
    }


run_check("Ex2", check_ex2)

Ex2: all supplied checks passed.


<a id="s03"></a>

# 3. Treat a function as a value

> **New requirement:** “Different usesr may want different results.”

We could write two almost identical reporting functions. Instead, we can supply the calculation itself as an argument. To see why this works, first separate a **function object** from the **result of calling it**.

In [301]:
calculation = mean
print("Calling the function through another name:", calculation([20.0, 22.0, 24.0]))

""" note, the max and min functions are available """
available_statistics = {"average": mean, "maximum": max, "minimum": min}
selected_statistic = available_statistics["maximum"]
print("Selected calculation:", selected_statistic([20.0, 22.0, 24.0]))

Calling the function through another name: 22.0
Selected calculation: 24.0


`mean` refers to a function. `mean([20.0, 22.0, 24.0])` calls it and produces a number. Parentheses make the difference. The dictionary stores functions, not precomputed answers, so a chosen function can later be used on a different room's measurements.

Functions are **first-class values**: we can assign them to names, store them in
containers, pass them as arguments, and return them. A **higher-order function** is a function that accepts another function or returns one. That formal term does not mean “more advanced”; it describes the shape of the inputs or result.

Python's [function-definition tutorial](https://docs.python.org/3.13/tutorial/controlflow.html#defining-functions) shows that a function definition creates a function object.

In [302]:
def calculate_for_room(room_name, values_c, statistic):
    value = statistic(values_c)
    return {"room": room_name, "result": value}


print(calculate_for_room("kitchen", [20.0, 22.0, 24.0], mean))
print(calculate_for_room("kitchen", [20.0, 22.0, 24.0], max))

{'room': 'kitchen', 'result': 22.0}
{'room': 'kitchen', 'result': 24.0}


The important contract is now partly about behaviour: `statistic` must be callable with one non-empty list or tuple of measurements and return the desired result. The reporting function does not need to know whether that result comes from `mean`, `max`, or a function written by us.

Passing a function is sometimes called passing a **callback** or a **strategy**. The useful idea is the same: the surrounding code stays fixed while one operation varies. This does not automatically guarantee purity. Passing a function that writes a file would introduce that side effect when it is called.

**Discussion:** what would happen if we passed `mean([20, 22])` instead of `mean`? We would pass a number, and the attempt to call that number inside the function would fail.

<a id="e03"></a>

## Ex3: One reporting function, several statistics

Write `describe_rooms(rooms, statistic)`. `rooms` is a dictionary mapping room names to
non-empty lists or tuples of finite Celsius values. Return a new dictionary mapping each
room to `statistic(values)`. Do not modify the input. An empty rooms dictionary should
produce an empty result.

Use the same function with `mean`, `max`, and a custom statistic. No special cases based
on function names are allowed: call the supplied function.

**Hint.** An ordinary loop that fills a newly created result dictionary is a clear starting point.

In [303]:
def describe_rooms(
    rooms: dict[str, list[float] | tuple[float, ...]],
    statistic: Callable[[list[float] | tuple[float, ...]], float],
) -> dict[str, float]:
    return {room: statistic(values) for room, values in rooms.items()}

In [304]:
def check_ex3():
    rooms = {"kitchen": [20.0, 22.0], "study": [23.0, 25.0]}
    snapshot = deepcopy(rooms)
    assert describe_rooms(rooms, mean) == {"kitchen": 21.0, "study": 24.0}
    assert describe_rooms(rooms, max) == {"kitchen": 22.0, "study": 25.0}

    def spread(values):
        return max(values) - min(values)

    assert describe_rooms(rooms, spread) == {"kitchen": 2.0, "study": 2.0}
    assert describe_rooms({}, mean) == {}
    assert rooms == snapshot


run_check("Ex3", check_ex3)

Ex3: all supplied checks passed.


<a id="s04"></a>

# 4. Small anonymous functions and meaningful names

> **New requirement:** “Show the warmest sensor first, without rearranging the archive.”

A sorting function already knows how to order things. What it does not know is which part of a sensor record matters. The `key` argument lets us supply that rule as a function. All readings in this example are already in Celsius.

In [305]:
latest_readings = [
    {"sensor_id": "study", "value": 24.0},
    {"sensor_id": "kitchen", "value": 21.0},
    {"sensor_id": "hall", "value": 22.5},
]


def reading_value(record):
    return record["value"]


ranked = sorted(latest_readings, key=reading_value, reverse=True)
print("Ranked:", ranked)
print("Original order:", latest_readings)

Ranked: [{'sensor_id': 'study', 'value': 24.0}, {'sensor_id': 'hall', 'value': 22.5}, {'sensor_id': 'kitchen', 'value': 21.0}]
Original order: [{'sensor_id': 'study', 'value': 24.0}, {'sensor_id': 'kitchen', 'value': 21.0}, {'sensor_id': 'hall', 'value': 22.5}]


When a function is this short and used only here, a `lambda` can express the rule inline.
`lambda record: record["value"]` means “a function taking `record` and returning its
`value` field.” The expression after the colon supplies the result; there is no `return`
statement. A lambda can contain only one expression, but that expression can still call
functions with side effects. Its syntax does not make it pure.

In [306]:
ranked = sorted(latest_readings, key=lambda record: record["value"], reverse=True)
print(ranked)

[{'sensor_id': 'study', 'value': 24.0}, {'sensor_id': 'hall', 'value': 22.5}, {'sensor_id': 'kitchen', 'value': 21.0}]


Use a named `def` when the operation has a meaningful domain name, needs explanation,
will be reused, or requires several steps. Use `lambda` for a small local rule whose
meaning is immediately visible. Compressing several decisions into a clever expression
usually makes a teaching example worse, not more functional.

`sorted` creates a new list; `a_list.sort()` changes the existing list. However, the
records inside the new sorted list are still the same dictionary objects. A new ordering
is not a deep copy. See the official
[`sorted` reference](https://docs.python.org/3.13/library/functions.html#sorted) and
[lambda tutorial](https://docs.python.org/3.13/tutorial/controlflow.html#lambda-expressions).

Two-part tuple keys let us break ties: Python compares the first component, then the
second when the first components match. For descending temperature but ascending sensor
name, `(-temperature, sensor_name)` keeps both decisions in one ascending key.

<a id="e04"></a>

## Ex4: Rank readings with a clear tie-break rule

Write `rank_readings(records)` to return a new list sorted by **decreasing** `value`, with equal temperatures ordered by **increasing** `sensor_id`. Assume every record has a finite Celsius `value` and a string `sensor_id`. Do not reorder the input list or edit its records. Returning references to the existing records in a new list is acceptable.

**Hint.** Supply `sorted` with a key that returns a two-element tuple: negative temperature, then sensor ID.

In [307]:
def rank_readings(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return sorted(records, key=lambda record: (-record["value"], record["sensor_id"]))

In [308]:
def check_e04():
    records = [
        {"sensor_id": "study", "value": 24.0},
        {"sensor_id": "hall", "value": 21.0},
        {"sensor_id": "kitchen", "value": 24.0},
    ]
    snapshot = deepcopy(records)
    result = rank_readings(records)
    assert [item["sensor_id"] for item in result] == ["kitchen", "study", "hall"]
    assert records == snapshot
    assert result is not records
    assert rank_readings([]) == []


run_check("E04", check_e04)

E04: all supplied checks passed.


<a id="s05"></a>

# 5. Transform, select, and summarise

> **New requirement:** “The new sensor uses Fahrenheit. Keep only readings that are too warm.”

Three questions often drive a data processing task. **Transform:** what should each measurement become? **Select:** which measurements should remain? **Summarise:** what result do we need from the remaining measurements? We will first keep these steps visible.

In [309]:
def f_to_c(value_f):
    """Convert a Fahrenheit value to Celsius"""
    return (value_f - 32.0) * 5.0 / 9.0


values_f = [68.0, 71.6, 77.0, 80.6]
values_c = [f_to_c(value) for value in values_f]
warm_values_c = [value for value in values_c if value >= 23.0]
print("Converted:", values_c)
print("Warm:", warm_values_c)

Converted: [20.0, 21.999999999999996, 25.0, 26.999999999999996]
Warm: [25.0, 26.999999999999996]


A **list comprehension** describes a new list. Read
`[f_to_c(value) for value in values_f]` from the loop outwards: take each value, convert it, and collect the result. An `if` clause keeps only items meeting a condition.

The same two operations have names in Python's functional toolkit. `map(function, items)` transforms items. `filter(predicate, items)` keeps items for which the supplied test is truthy. A **predicate** is a function used to express a true/false question. We will write predicates that explicitly return `True` or `False`.

In [310]:
def at_least_23(value_c):
    return value_c >= 23.0


converted = map(f_to_c, values_f)
selected = filter(at_least_23, converted)
print(list(selected))

[25.0, 26.999999999999996]


In Python 3, `map` and `filter` return **iterators**, not finished lists. `list(selected)` asks for all their results and stores them. We will examine that delayed computation later on in Section 9. For now, use `list(...)` when comparing a finite example with a comprehension.

See the [`map`](https://docs.python.org/3.13/library/functions.html#map) and
[`filter`](https://docs.python.org/3.13/library/functions.html#filter) reference entries.

Neither usecase is automatically better. A comprehension often makes a combined
transformation and condition easy to read. `map` is convenient when a good named
transformation already exists. A loop is a good choice when the work needs several explained steps or detailed error handling. All three can avoid changing the input.

In [311]:
""" Three different implementations of the same transformation """
with_loop = []
for value_f in values_f:
    with_loop.append(f_to_c(value_f))

with_comprehension = [f_to_c(value_f) for value_f in values_f]
with_map = list(map(f_to_c, values_f))
assert with_loop == with_comprehension == with_map

print("Any warm reading?", any(value >= 23.0 for value in with_map))
print("All readings warm?", all(value >= 23.0 for value in with_map))
print("Warm-reading count:", sum(value >= 23.0 for value in with_map))

Any warm reading? True
All readings warm? False
Warm-reading count: 2


The threshold is in Celsius, so conversion must happen before that comparison. Filtering raw Fahrenheit values against `23.0` would run without an exception but answer the wrong question. Small functions help us inspect the steps; they do not protect us from choosing the wrong units or policy.

`any` asks whether at least one condition is true; `all` asks whether every condition is true. Both can stop as soon as the answer is known. For an empty input, `any` returns `False` and `all` returns `True`. That last result does **not** prove a sensor is working: “all received readings passed” and “we received a reading” are different checks.

<a id="e05"></a>

## Ex5: Compare two versions of the same pipeline

Implement two functions with identical behaviour:
`warm_celsius_comp(values_f, threshold_c)` using comprehensions, and
`warm_celsius_tools(values_f, threshold_c)` using `map` and `filter`.
Both must return a list of Celsius values at or above the threshold, in input order. Assume a finite list or tuple of finite Fahrenheit values. Empty input returns `[]`. Do not change the input.

**Hint.** The two-stage version is easier to read than a one-liner that converts the same value twice. A small lambda can use the current `threshold_c` argument.

In [312]:
def warm_celsius_comp(
    values_f: list[float] | tuple[float, ...], threshold_c: float
) -> list[float]:
    return [f_to_c(value_f) for value_f in values_f if f_to_c(value_f) >= threshold_c]


def warm_celsius_tools(
    values_f: list[float] | tuple[float, ...], threshold_c: float
) -> list[float]:
    return list(
        map(f_to_c, filter(lambda value_f: f_to_c(value_f) >= threshold_c, values_f))
    )

In [313]:
def check_ex5():
    values = [68.0, 73.4, 77.0, 80.6]
    snapshot = values.copy()
    expected = [23.0, 25.0, 27.0]
    for function in (warm_celsius_comp, warm_celsius_tools):
        result = function(values, 23.0)
        assert isinstance(result, list)
        assert len(result) == len(expected)
        assert all(isclose(actual, wanted) for actual, wanted in zip(result, expected))
        assert function([], 23.0) == []
        assert function([32.0], 10.0) == []
    assert values == snapshot


run_check("Ex5", check_ex5)

Ex5: all supplied checks passed.


<a id="s06"></a>

# 6. Build a function with its own policy: **closures**

> **New requirement:** “The kitchen and the nursery need different alert thresholds.”

Passing the threshold on every call is perfectly good. Sometimes we want to configure a
rule once and pass that ready-to-use rule to another part of the program. A function can
return another function for this purpose.

In [314]:
def make_hot_rule(threshold_c):
    def check(value_c):
        return value_c >= threshold_c

    """ NOTE! Return the function, not the result of calling it """
    return check


kitchen_rule = make_hot_rule(23.0)
nursery_rule = make_hot_rule(21.0)
print("Kitchen:", kitchen_rule(22.0))
print("Nursery:", nursery_rule(22.0))
print("Nursery alerts:", list(filter(nursery_rule, [20.0, 21.5, 22.0])))

Kitchen: False
Nursery: True
Nursery alerts: [21.5, 22.0]


Walk through the two different calls. `make_hot_rule(23.0)` creates a function called `check` and returns it. It does **not** check a measurement yet. The returned function retains access to the `threshold_c` belonging to that factory call. Later, `kitchen_rule(22.0)` supplies the measurement and performs the comparison.

A **closure** is a function together with access to variables from its enclosing scope.
A **factory function** is simply a function that creates something; here it creates a configured function. Each call above creates a separate enclosing context. Because our thresholds are immutable numbers and we do not reassign them, these particular rules have stable behaviour.

Retained access is not a frozen snapshot of every object. Capturing a dictionary that other code later edits can change a closure's answers. Closures can also maintain changing state using `nonlocal`; that may be useful, but it is not the same design as a pure value calculation.

### A small trap: creating functions inside a loop

In [315]:
bad_rules = [lambda value_c: value_c >= limit for limit in (21.0, 23.0, 25.0)]
print("Unexpected rules:", [rule(24.0) for rule in bad_rules])

good_rules = [make_hot_rule(limit) for limit in (21.0, 23.0, 25.0)]
print("Independent rules:", [rule(24.0) for rule in good_rules])

Unexpected rules: [False, False, False]
Independent rules: [True, True, False]


The first functions look up the same enclosing `limit` when called, after the
comprehension has finished. At that point its last value is 25.0. This is called
**late binding**. It is not a bug specific to lambda syntax; a nested `def` can behave the
same way. The factory version gives each rule its own call context. Another common fix
is a default argument such as `lambda value_c, limit=limit: value_c >= limit`; the factory
is easier to explain when the rule needs a name. See the
[Python FAQ on functions defined in a loop](https://docs.python.org/3.13/faq/programming.html#why-do-lambdas-defined-in-a-loop-with-different-values-all-return-the-same-result).

<a id="e06"></a>

## Ex6 — Create a comfort-band rule

Write `make_band_checker(lower_c, upper_c)`. It must return a function that says whether
a measurement lies inside the **inclusive** interval from `lower_c` to `upper_c`.
The returned function must return a Boolean. A single-point interval is allowed.
Raise `ValueError` immediately when `lower_c > upper_c`. Assume finite numerical bounds
and measurements.

Create separate kitchen and study checkers; making the second must not change the first.

**Hint.** Validate the configuration in the outer function. Put the measurement comparison in a nested function, then return that function.

In [316]:
def make_band_checker(lower_c, upper_c):
    if lower_c > upper_c:
        raise ValueError("lower_c must not be greater than upper_c.")

    def in_band(value_c):
        return lower_c <= value_c <= upper_c

    return in_band

In [317]:
def check_e06():
    kitchen_ok = make_band_checker(20.0, 23.0)
    study_ok = make_band_checker(22.0, 25.0)
    assert kitchen_ok(20.0) is True
    assert kitchen_ok(23.0) is True
    assert kitchen_ok(23.1) is False
    assert study_ok(24.0) is True
    assert kitchen_ok(24.0) is False
    assert make_band_checker(21.0, 21.0)(21.0) is True
    assert_raises(ValueError, make_band_checker, 24.0, 20.0)


run_check("E06", check_e06)

E06: all supplied checks passed.


<a id="s07"></a>

# 7. Configure and connect functions

> **New requirement:** “The replacement sensor needs unit conversion and its own calibration.”

A closure is useful when we are creating a new rule. When a suitable multi-argument
function already exists, `functools.partial` can supply some of its arguments in advance.
This operation is called **partial application**.

In [318]:
def calibrate(value_c, gain=1.0, offset_c=0.0):
    """Apply a simple illustrative calibration in Celsius."""
    return gain * value_c + offset_c


kitchen_calibration = partial(calibrate, gain=1.02, offset_c=-0.4)
print(kitchen_calibration(20.0))
print(list(map(kitchen_calibration, [20.0, 21.0, 22.0])))

20.0
[20.0, 21.020000000000003, 22.040000000000003]


Calling `kitchen_calibration(20.0)` supplies the remaining measurement argument, while `gain` and `offset_c` come from the configuration. The measurement should not be supplied when we build the partial; it varies from reading to reading.

`partial` creates a callable, not an immutable policy object. Stored keyword arguments can be overridden by keywords in a later call, and referenced mutable objects are not deep-copied. In this example we call it with only the measurement. See the [`partial` reference](https://docs.python.org/3.13/library/functools.html#functools.partial).

You may also hear **currying**. That is a related but different transformation: turning a multi-argument function into a chain of one-argument functions. We are simply supplying some arguments in advance; no currying machinery is required. [PEP 309](https://peps.python.org/pep-0309/) discusses the distinction.

### **Composition**: use one function's result as another's input

For this sensor the required order is Fahrenheit → Celsius → calibration in Celsius. Start with the direct spelling. Names for intermediate values make debugging easy.

In [319]:
raw_f = 77.0
converted_c = f_to_c(raw_f)
corrected_c = kitchen_calibration(converted_c)
print("After conversion:", converted_c)
print("After calibration:", corrected_c)

# The nested spelling describes the same order: inner call first.
assert corrected_c == kitchen_calibration(f_to_c(raw_f))

After conversion: 25.0
After calibration: 25.1


**Function composition** means connecting functions so that one function's output
becomes another's input. We do not need a library to do it. A small helper is useful only
when we want to create and reuse the combined operation. Here is its complete definition.

In [320]:
def compose(after, before):
    """Return a one-argument function that applies before, then after."""

    def combined(value):
        return after(before(value))

    return combined


prepare_kitchen_value = compose(kitchen_calibration, f_to_c)
print(list(map(prepare_kitchen_value, [68.0, 77.0])))

[20.0, 25.1]


Our helper deliberately handles just two one-argument functions. Its contract is
visible: `before` must produce something that `after` accepts. Reversing the functions
would calibrate a Fahrenheit number as though it were Celsius. Composition is not
usually commutative: order matters.

The nested `combined` function does not make impure components pure. It simply connects
them. Likewise, a long chain of small functions can still be difficult to understand.
Prefer the version whose intermediate meanings a student or teammate can explain.

<a id="e07"></a>

## Ex7: Create a calibrated Fahrenheit reader

Write `make_calibration_pipeline(gain, offset_c)` using `partial` and `compose`.
Return a one-argument callable that first converts Fahrenheit to Celsius and then applies `gain * value_c + offset_c`. Assume finite numerical inputs and settings.

For `gain=1.1` and `offset_c=-1.0`, an input of 68 °F should become 21 °C.
The settings are supplied when creating the pipeline, not on every reading.

**Hint.** Partially apply `calibrate`, then place that configured function after `f_to_c` in the composition.

In [321]:
def make_calibration_pipeline(gain, offset_c):
    calibrated = partial(calibrate, gain=gain, offset_c=offset_c)
    return compose(calibrated, f_to_c)

In [322]:
def check_e07():
    prepare = make_calibration_pipeline(1.1, -1.0)
    assert callable(prepare)
    assert isclose(prepare(68.0), 21.0)
    assert isclose(prepare(77.0), 26.5)
    identity_calibration = make_calibration_pipeline(1.0, 0.0)
    assert isclose(identity_calibration(32.0), 0.0)
    assert isclose(prepare(68.0), 21.0)


run_check("E07", check_e07)

E07: all supplied checks passed.


<a id="s08"></a>

# 8. Reduce many readings to a small state

> **New requirement:** “Keep a summary while processing readings, not a separate answer for each one.”

A transformation gives one output per input. A **reduction** combines many inputs into one result. `sum`, `min`, `max`, `any`, and `all` are familiar examples. Use these clear, specialized tools when they already say what you need.

For a less standard summary, define a **state** and a **step**. Our state below is a tuple `(count, total_c)`. The step receives the old state and one new temperature, and returns a new state. It does not edit the old one.

In [323]:
def advance_totals(state, value_c):
    count_so_far, total_c = state
    return (count_so_far + 1, total_c + value_c)


state = (0, 0.0)
print("Start:", state)
for value_c in [20.0, 22.0, 24.0]:
    state = advance_totals(state, value_c)
    print("After", value_c, "→", state)

Start: (0, 0.0)
After 20.0 → (1, 20.0)
After 22.0 → (2, 42.0)
After 24.0 → (3, 66.0)


The changing local name `state` refers to each new tuple in turn. That is different from
silently changing a shared object. To find the average at the end, divide the total by the count, provided the count is not zero.

`functools.reduce` performs this left-to-right stepping for us and returns the final state. The explicit initial state also defines what happens for empty input. See the [`reduce` reference](https://docs.python.org/3.13/library/functools.html#functools.reduce).

In [324]:
final_state = reduce(advance_totals, [20.0, 22.0, 24.0], (0, 0.0))
empty_state = reduce(advance_totals, [], (0, 0.0))
print("Final state:", final_state)
print("Average:", final_state[1] / final_state[0])
print("Empty input:", empty_state)

Final state: (3, 66.0)
Average: 22.0
Empty input: (0, 0.0)


The reduction is the same sequence of calls as the visible loop; it is not a parallel algorithm. In general, changing grouping or order can change a calculation. Subtraction is an obvious example, and even floating-point addition can depend slightly on order. Do not infer “safe to parallelize” merely because an operation uses `reduce`.

For a plain total, `sum(values)` is clearer than `reduce(lambda a, b: a + b, values, 0)`. A reduction becomes useful when the state has a meaningful structure, such as a count, a total, and an alert count. Do not use a reduction to repeatedly concatenate growing lists just to avoid an ordinary loop; that can involve unnecessary copying.

<a id="e08"></a>

## Ex8: Reduce readings to count, sum, and alert count

Implement `accumulate_temperature(state, value_c, threshold_c=23.0)` where state is `(count, total_c, hot_count)`. Return a new tuple after adding the reading; a reading counts as hot when it is at or above the threshold.

Then implement `temperature_totals(values_c, threshold_c=23.0)` using `reduce` and a configured step. Return `(0, 0.0, 0)` for empty input. Assume a finite iterable of finite Celsius numbers. Consuming a supplied iterator is part of this streaming interface.

**Hint.** Unpack the three fields, return three updated fields, and use `partial` to configure the threshold without changing the two-argument reduction interface.

In [325]:
def accumulate_temperature(
    state: tuple, value_c: float, threshold_c: float = 23.0
) -> tuple[int, float | int, int]:
    count_so_far, total_c, hot_count = state
    new_count = count_so_far + 1
    new_total = total_c + value_c
    new_hot_count = hot_count + (1 if value_c >= threshold_c else 0)
    return new_count, new_total, new_hot_count


def temperature_totals(values_c, threshold_c=23.0):
    step = partial(accumulate_temperature, threshold_c=threshold_c)
    return reduce(step, values_c, (0, 0.0, 0))

In [326]:
def check_ex8():
    old_state = (2, 42.0, 0)
    assert accumulate_temperature(old_state, 24.0, 23.0) == (3, 66.0, 1)
    assert old_state == (2, 42.0, 0)
    assert temperature_totals([20.0, 23.0, 25.0], 23.0) == (3, 68.0, 2)
    assert temperature_totals([20.0, 23.0, 25.0], 25.0) == (3, 68.0, 1)
    assert temperature_totals([], 23.0) == (0, 0.0, 0)
    assert temperature_totals(iter([20.0, 24.0]), 23.0) == (2, 44.0, 1)


run_check("Ex8", check_ex8)

Ex8: all supplied checks passed.


<a id="s09"></a>

# 9. Process a stream without building every result first

> **New requirement:** “The sensor keeps sending readings. There is no final list to wait for.”

A list stores items we can revisit. An **iterable** is something from which iteration can
begin. An **iterator** remembers a current position and supplies successive items through
`next`. A list is iterable; calling `iter` on it creates an iterator over the list.

In [327]:
values = [20.0, 21.0, 22.0]
cursor = iter(values)
print("First:", next(cursor))
print("Second:", next(cursor))
print("Remaining:", list(cursor))
print("Remaining again:", list(cursor))
print("A fresh pass over the list:", list(iter(values)))

First: 20.0
Second: 21.0
Remaining: [22.0]
Remaining again: []
A fresh pass over the list: [20.0, 21.0, 22.0]


An iterator is normally **one-shot**. The empty second result does not mean the sensor had no measurements; it means this cursor has already consumed them. Advancing a cursor changes its state. Iterator pipelines are useful in functional-style Python, but an iterator object is not an immutable collection or a pure mathematical value.

**Lazy evaluation** here means computing results as they are requested. A generator expression uses parentheses instead of a list comprehension's square brackets. This lets us process a large source without storing all transformed outputs at once, though the source and downstream consumers may still use substantial memory.

In [328]:
# This helper prints deliberately, only to make evaluation timing visible.
def traced_conversion(value_f):
    print("Converting", value_f)
    return f_to_c(value_f)


lazy_values = (traced_conversion(value) for value in [68.0, 77.0])
print("Generator created; no conversions printed yet.")
print("First requested result:", next(lazy_values))
print("Rest:", list(lazy_values))

Generator created; no conversions printed yet.
Converting 68.0
First requested result: 20.0
Converting 77.0
Rest: [25.0]


The print inside `traced_conversion` is instrumentation, not an example of purity.
Delaying work does not remove its side effects. A failure inside a transformation may
also be delayed until iteration. The leftmost iterable in a generator expression is
prepared when the expression is created; laziness does not mean that literally nothing
at all happens at creation. See the
[generator-expression reference](https://docs.python.org/3.13/reference/expressions.html#generator-expressions).

### Write a generator with `yield`

A generator function can express several steps between outputs. Calling it creates a
generator object. `yield` supplies one value and pauses the body; the next request resumes
from that point. `return` ends it. There is no automatic thread, background task, or
real-time scheduling involved. The [Python tutorial on generators](https://docs.python.org/3.13/tutorial/classes.html#generators)
explains this resumable execution model.

In [329]:
def example_sensor_stream(start_c=20.0, step_c=0.25):
    value_c = start_c
    while True:
        yield value_c
        value_c += step_c


stream = example_sensor_stream()
first_five = list(islice(stream, 5))
print("First five:", first_five)
print("Next reading:", next(stream))

First five: [20.0, 20.25, 20.5, 20.75, 21.0]
Next reading: 21.25


This deliberately simple simulator can supply as many readings as requested. It is not a
physical temperature model. Never call `list(stream)` on this unbounded generator.
`islice(stream, 5)` limits consumption to five source items.

For an alert stream, choose where the limit belongs. “Inspect the next 100 measurements”
is a bounded task. “Wait until five alerts arrive” may never finish if none qualifies.
The following placement limits the **input before filtering**, so it terminates even
when the output is empty. This bounds item count, not the wall-clock duration of a real
sensor read that could itself block.

In [330]:
bounded_source = islice(example_sensor_stream(), 100)
alerts = filter(make_hot_rule(1000.0), bounded_source)
print("Alerts among the next 100 readings:", list(alerts))

# Do not run this on an unbounded source that may never produce an alert:
# list(islice(filter(alert_rule, unbounded_source), 5))

Alerts among the next 100 readings: []


Short-circuit consumers also move the cursor. After `any` finds a match, items before and
including that match have been consumed; later items remain. To process the same finite
stream twice, either create a fresh source or intentionally materialize it once. Do not
silently reuse an exhausted iterator for a second report.

<a id="e09"></a>

## Ex9: Calibrate lazily

Write the generator function `iter_calibrated(values_c, offset_c)`. It should yield each input value plus the offset, in order. It must not collect the input in a list or consume future values before they are requested. Empty input yields nothing.

Assume finite numerical values, but the iterable itself may be unbounded. The function therefore must work one item at a time.

**Hint.** Use a `for` loop and `yield`. The caller, not the generator, decides when to stop requesting values.

In [331]:
def iter_calibrated(values_c, offset_c):
    for value in values_c:
        yield value + offset_c

In [332]:
def check_ex9():
    seen = []

    def source():
        for value in (20.0, 21.0, 22.0):
            seen.append(value)  # Instrumentation, not part of the transformation.
            yield value

    output = iter_calibrated(source(), 0.5)
    assert iter(output) is output
    assert seen == []
    assert isclose(next(output), 20.5)
    assert seen == [20.0]
    assert list(output) == [21.5, 22.5]
    assert list(output) == []
    assert list(iter_calibrated([], 0.5)) == []
    assert list(islice(iter_calibrated(count(20), 0.5), 3)) == [20.5, 21.5, 22.5]


run_check("Ex9", check_ex9)

Ex9: all supplied checks passed.


<a id="s10"></a>

# 10. A few useful stream building blocks

> **New requirement:** “Show cumulative information while readings arrive.”

`itertools` supplies small iterator operations that fit together. We will use only a few
whose purpose is easy to see. `chain` joins sources in sequence, `pairwise` exposes
neighbouring values, and `accumulate` returns each successive summary rather than only
the final one. Their exact interfaces are documented in the
[`itertools` reference](https://docs.python.org/3.13/library/itertools.html).

In [333]:
morning_c = [20.0, 21.0]
afternoon_c = [23.0, 22.0]
print("Joined readings:", list(chain(morning_c, afternoon_c)))

values_c = [20.0, 21.0, 23.0, 22.0]
changes_c = [current - previous for previous, current in pairwise(values_c)]
print("Changes between consecutive readings:", changes_c)

interval_energy_kwh = [0.10, 0.15, 0.05]
print("Energy used so far:", list(accumulate(interval_energy_kwh)))

Joined readings: [20.0, 21.0, 23.0, 22.0]
Changes between consecutive readings: [1.0, 2.0, -1.0]
Energy used so far: [0.1, 0.25, 0.3]


`chain` does not merge by timestamp; it finishes the first iterable before starting the second. `pairwise` gives one fewer pair than the number of readings. Its differences are changes per sample; turning them into a rate would require elapsed times.

`accumulate` without a custom step uses addition. With a step, it can track the count and total state from the previous section. With an explicit initial state it also yields that state, before it consumes the first reading. In contrast, `reduce` returns only the final state.

In [334]:
states = list(accumulate([20.0, 22.0, 24.0], advance_totals, initial=(0, 0.0)))
print("Initial and successive states:", states)
print("Running averages:", [total / count_ for count_, total in states if count_ > 0])

Initial and successive states: [(0, 0.0), (1, 20.0), (2, 42.0), (3, 66.0)]
Running averages: [20.0, 21.0, 22.0]


A running average describes **all readings seen so far**. It is not a moving average over
a fixed-size recent window. A recent-window calculation needs to remember which old
value leaves the window; our two-number state does not contain that information.

### Optional caution: `groupby` groups adjacent items

The name can be misleading. `itertools.groupby` starts a new group whenever the key
changes. It does not collect all equal keys scattered across the source. For a global
per-room grouping, sort by room first, then consume each group before moving to the next.
Sorting materializes the input, so this is not an unbounded streaming operation.

In [335]:
room_samples = [("kitchen", 20.0), ("study", 24.0), ("kitchen", 22.0)]
by_room = lambda item: item[0]

print("Adjacent groups:")
for room, group in groupby(room_samples, key=by_room):
    print(room, list(group))

print("After sorting by room:")
for room, group in groupby(sorted(room_samples, key=by_room), key=by_room):
    print(room, list(group))

Adjacent groups:
kitchen [('kitchen', 20.0)]
study [('study', 24.0)]
kitchen [('kitchen', 22.0)]
After sorting by room:
kitchen [('kitchen', 20.0), ('kitchen', 22.0)]
study [('study', 24.0)]


The loop consumes `group` immediately because it shares the underlying iterator with
`groupby`. Storing group iterators to use after the outer iteration has advanced is a
common source of empty or incomplete groups. These details matter more than memorizing
many function names.

<a id="e10"></a>

## Ex10: Yield a running mean

Write the generator function `running_mean(values_c)` using `accumulate` and
`advance_totals`. For inputs 20, 22, and 24, it must yield 20, 21, and 22.
Empty input must yield nothing. Keep only the small running state; do not collect all measurements. Inputs are finite numbers, but the iterable may be unbounded.

Skip the initial zero-count state rather than dividing by zero.

**Hint.** Use `initial=(0, 0.0)`. For each emitted state with a positive count, yield total divided by count.

In [336]:
from typing import Iterable


def running_mean(values_c: Iterable[float]) -> Iterable[float]:
    for count_, total in accumulate(values_c, advance_totals, initial=(0, 0.0)):
        if count_ > 0:
            yield total / count_

In [337]:
def check_ex10():
    assert list(running_mean([20.0, 22.0, 24.0])) == [20.0, 21.0, 22.0]
    assert list(running_mean([])) == []
    assert list(running_mean([0.0])) == [0.0]
    assert list(running_mean([-2.0, 2.0])) == [-2.0, 0.0]
    stream = running_mean(count(1))
    assert list(islice(stream, 4)) == [1.0, 1.5, 2.0, 2.5]


run_check("Ex10", check_ex10)

Ex10: all supplied checks passed.


<a id="s11"></a>

# 11. Keep effects at the edges and a clear contract in the middle

> **New requirement:** “A bad message must not crash the whole report or disappear without explanation.”

Our early functions assumed good numbers. A real boundary can deliver missing fields, unsupported units, failed-sensor status, or a value such as NaN. Before a calculation uses a message, we need a **data contract**: the fields and meanings we agree to accept.

For this classroom project, one sensor ID is also the room label. We use an integer `minute` from the start of a simulated run, rather than introducing timestamp parsing.

A deployed system would need a separately agreed timestamp and identity contract.

| Raw field | Classroom contract |
|:--|:--|
| `sensor_id` | A non-empty, non-whitespace string. |
| `minute` | A non-negative built-in integer; a Boolean is not accepted. |
| `value` | A finite built-in integer or float; not a Boolean. |
| `unit` | Exactly `"C"` or `"F"`. |
| `status` | Exactly `"ok"` for a reading we may use. |
| Physical range | The value must correspond to −50 through 60 °C, inclusive. This is our illustrative acceptance policy, not a universal sensor specification. |

Accepted output records have just `sensor_id`, `minute`, and `temperature_c`.
The changed field name makes the normalized unit visible. Rejected input positions and
reasons will be returned separately. We will not quietly turn invalid values into zero:
zero is a real measurement, not a synonym for “unknown.”

The validator below returns `None` for an accepted message, otherwise the first reason
it cannot be accepted. It checks structure before attempting a conversion.

In [338]:
def validation_error(record):
    """Return the first contract violation, or None for an accepted reading."""
    if not isinstance(record, dict):
        return "record must be a dictionary"
    sensor_id = record.get("sensor_id")
    if not isinstance(sensor_id, str) or not sensor_id.strip():
        return "sensor_id must be a non-empty string"
    minute = record.get("minute")
    if type(minute) is not int or minute < 0:
        return "minute must be a non-negative integer"
    if record.get("status") != "ok":
        return "sensor status is not ok"
    value = record.get("value")
    if type(value) not in (int, float):
        return "value must be a number, not a Boolean"
    if type(value) is float and not isfinite(value):
        return "value must be finite"
    unit = record.get("unit")
    if unit not in ("C", "F"):
        return "unit must be C or F"
    lower_raw, upper_raw = (-50.0, 60.0) if unit == "C" else (-58.0, 140.0)
    if not lower_raw <= value <= upper_raw:
        return "temperature is outside the accepted range"
    return None

A Boolean is a special case of an integer in Python, so the explicit `type` checks here
are intentional. This narrow contract accepts built-in numerical types, not every
possible numerical-library scalar. `isfinite` rejects NaN and infinity; checking raw
range limits before conversion also avoids converting an arbitrarily huge integer to a
float. See [`math.isfinite`](https://docs.python.org/3.13/library/math.html#math.isfinite).

Validation and normalization are distinct operations. The next function assumes the
validator has accepted the record. It creates a new, simpler dictionary; the original
message remains available for an audit. Treat that precondition as part of its contract.

In [339]:
def normalise_reading(record):
    """Convert an already validated raw record to the canonical schema."""
    value_c = record["value"] if record["unit"] == "C" else f_to_c(record["value"])
    return {
        "sensor_id": record["sensor_id"],
        "minute": record["minute"],
        "temperature_c": float(value_c),
    }

In [340]:
example_records = [
    {"sensor_id": "kitchen", "minute": 0, "value": 21.0, "unit": "C", "status": "ok"},
    {"sensor_id": "study", "minute": 0, "value": 77.0, "unit": "F", "status": "ok"},
    {"sensor_id": "kitchen", "minute": 1, "value": 23.0, "unit": "C", "status": "ok"},
    {"sensor_id": "study", "minute": 1, "value": 72.5, "unit": "F", "status": "ok"},
    {"sensor_id": "garage", "minute": 0, "value": None, "unit": "C", "status": "ok"},
    {
        "sensor_id": "kitchen",
        "minute": 2,
        "value": float("nan"),
        "unit": "C",
        "status": "ok",
    },
    {"sensor_id": "study", "minute": 2, "value": 24.0, "unit": "C", "status": "fault"},
    {"sensor_id": "spare", "minute": 0, "value": 295.0, "unit": "K", "status": "ok"},
]


for index, record in enumerate(example_records):
    reason = validation_error(record)
    result = normalise_reading(record) if reason is None else reason
    print(index, "→", result)

0 → {'sensor_id': 'kitchen', 'minute': 0, 'temperature_c': 21.0}
1 → {'sensor_id': 'study', 'minute': 0, 'temperature_c': 25.0}
2 → {'sensor_id': 'kitchen', 'minute': 1, 'temperature_c': 23.0}
3 → {'sensor_id': 'study', 'minute': 1, 'temperature_c': 22.5}
4 → value must be a number, not a Boolean
5 → value must be finite
6 → sensor status is not ok
7 → unit must be C or F


### Functional core, imperative shell

A useful arrangement for this project is:

```text
receive messages → validate and calculate → display a report or send a command
     boundary        functional core                 boundary
```

The **functional core** makes decisions from explicit data. The **imperative shell**
handles interactions: reading, writing, displaying, and calling devices. This is our
design vocabulary, not a special Python feature. Pure functions can still reject input
by raising a predictable exception; alternatively, a batch-processing function can return
accepted results and rejection reasons as data, as we do here.

Even report formatting can be separated from displaying it. The following function only
returns text. `print` is deliberately outside it. Replacing printing with a web response
would not require rewriting the summary calculation.

In [341]:
def format_summary(room_name, summary):
    return (
        f"{room_name}: mean {summary['mean_c']:.2f} °C; "
        f"{summary['hot_count']} of {summary['count']} readings at or above the threshold"
    )


message = format_summary("kitchen", summarise([21.0, 23.0], threshold_c=23.0))
print(message)

kitchen: mean 22.00 °C; 1 of 2 readings at or above the threshold


A pure core does not eliminate input/output, and an ordinary function is not automatically
part of the pure core just because it has a `def`. Ask what each operation depends on
and what it changes. Later, a small wrapper could accept a `send_message` function and
call it with the returned text, which would let us test the wrapper with a recording
function instead of contacting a real device.

<a id="e11"></a>

## Ex11: Keep accepted readings and explain rejected ones

Write `prepare_batch(raw_records)` using the supplied validator and normalizer. The input
is a finite list or tuple. Return a dictionary with two lists:

```python
{
    "accepted": [canonical_record, ...],
    "rejected": [{"index": original_position, "reason": error_text}, ...],
}
```

Keep the original relative order within each list. Every input must appear exactly once
as either an accepted reading or a rejection. Rejection indices are zero-based. Do not
modify input records. Empty input returns two empty lists. Do not call the normalizer
on rejected input, and do not print inside this function.

**Hint.** A loop with two new local lists is particularly clear here. Evaluate the validator once per record; branch on whether the result is `None`.

In [342]:
def prepare_batch(raw_records: list[dict]) -> dict[str, list]:
    accepted = []
    rejected = []

    for i, rec in enumerate(raw_records):
        reason = validation_error(rec)
        if reason is None:
            accepted.append(normalise_reading(rec))
        else:
            rejected.append({"index": i, "reason": reason})

    return {"accepted": accepted, "rejected": rejected}

In [343]:
def check_ex11():
    records = [
        {"sensor_id": "a", "minute": 0, "value": 20.0, "unit": "C", "status": "ok"},
        {"sensor_id": "b", "minute": 1, "value": 77.0, "unit": "F", "status": "ok"},
        {"sensor_id": "a", "minute": 2, "value": None, "unit": "C", "status": "ok"},
        {"sensor_id": "b", "minute": 3, "value": 24.0, "unit": "C", "status": "fault"},
    ]
    snapshot = deepcopy(records)
    result = prepare_batch(records)
    assert result["accepted"] == [
        {"sensor_id": "a", "minute": 0, "temperature_c": 20.0},
        {"sensor_id": "b", "minute": 1, "temperature_c": 25.0},
    ]
    assert result["rejected"] == [
        {"index": index, "reason": validation_error(records[index])} for index in (2, 3)
    ]
    assert len(result["accepted"]) + len(result["rejected"]) == len(records)
    assert records == snapshot
    assert result["accepted"][0] is not records[0]
    assert prepare_batch([]) == {"accepted": [], "rejected": []}
    invalid = [
        {},
        None,
        {**records[0], "value": True},
        {**records[0], "value": float("nan")},
    ]
    assert len(prepare_batch(invalid)["rejected"]) == 4
    assert prepare_batch(invalid)["accepted"] == []


run_check("Ex11", check_ex11)

Ex11: all supplied checks passed.


<a id="s12"></a>

# 12. State can be an explicit input, too

> **New requirement:** “The simulated heater should not switch on and off whenever noise crosses one threshold.”

Functional programming does not require pretending that a device has no state. It asks
us to make state visible when it matters. We can write a transition as
`new_state = transition(old_state, measurement, settings)`. That transition can be a pure
calculation even though the application evolves over time.

For our teaching controller we propose this policy: turn heating on at or below a lower
bound; turn it off at or above an upper bound; between the bounds, keep the previous
state. The bounds must be distinct. This two-threshold, history-dependent rule is an
example of **hysteresis**. It is a small software model, not a controller ready for a real
heating installation. Hardware safety and communication failures are separate concerns.

Compare that requirement with a single-threshold rule. The following measurements hover
around 21 °C, so a simple rule repeatedly changes its decision.

For background on the two-threshold idea, see the [MathWorks relay description](https://www.mathworks.com/help/simulink/slref/relay.html). Its generic relay switches in the opposite direction to our heating policy; our exact rule is stated here and tested in E12.

In [344]:
noisy_values_c = [20.9, 21.1, 20.8, 21.2, 20.9, 21.1]
single_threshold_states = [value < 21.0 for value in noisy_values_c]
print("Measurements:", noisy_values_c)
print("Single-threshold decisions:", single_threshold_states)

Measurements: [20.9, 21.1, 20.8, 21.2, 20.9, 21.1]
Single-threshold decisions: [True, False, True, False, True, False]


### Test the rule, not just one demonstration

Pure value calculations let us supply the entire relevant situation directly. For the
new rule, useful tests include a very cold room, a very warm room, both exact boundaries,
an in-between measurement with heating previously off, the same measurement with heating
previously on, and invalid bounds.

Properties connect these examples. Below the lower boundary, the old state should not
matter. Inside the interval, it should matter. Testing both histories is essential;
otherwise a function that ignores `was_on` could pass a superficial demonstration.

These checks describe behaviour rather than implementation. A loop, a comprehension, or
a helper function may all be valid implementations when they meet the same contract.
Assertions are for our tests, not user-input validation: use an explicit exception for
invalid configuration. See the [assert statement reference](https://docs.python.org/3.13/reference/simple_stmts.html#the-assert-statement).

<a id="e12"></a>

## Ex12: Make the thermostat transition pure

Write `next_heating_state(was_on, temperature_c, lower_c=20.0, upper_c=22.0)`.
Return `True` at or below `lower_c`, return `False` at or above `upper_c`, and otherwise return the supplied previous Boolean state. Raise `ValueError` when `lower_c >= upper_c`.
Assume finite numerical temperatures and bounds and a Boolean previous state.

The function must not remember a previous call, change a global variable, or operate a heater. Previous state is an input; proposed next state is the result.

**Hint.** Validate first. Handle the cold and warm cases explicitly, then let the remaining case return `was_on`.

In [345]:
def next_heating_state(
    was_on: bool,
    temperature_c: float,
    lower_c: float = 20.0,
    upper_c: float = 22.0,
) -> bool:
    if lower_c >= upper_c:
        raise ValueError("lower_c must be less than upper_c")

    if temperature_c <= lower_c:
        return True
    if temperature_c >= upper_c:
        return False
    return was_on

In [346]:
def check_ex12():
    for previous in (False, True):
        assert next_heating_state(previous, 19.0) is True
        assert next_heating_state(previous, 20.0) is True
        assert next_heating_state(previous, 22.0) is False
        assert next_heating_state(previous, 23.0) is False
        assert next_heating_state(previous, 21.0) is previous
    assert_raises(ValueError, next_heating_state, False, 21.0, 22.0, 20.0)
    assert_raises(ValueError, next_heating_state, False, 21.0, 21.0, 21.0)
    assert next_heating_state(False, 21.0) is False
    assert next_heating_state(True, 21.0) is True


run_check("Ex12", check_ex12)

Ex12: all supplied checks passed.


<a id="s13"></a>

# 13. Capstone: a room report from a simulated sensor feed

> **New requirement:** “Demonstrate the full route from incoming records to a useful dashboard.”

Before the capstone exercise, let us run a complete **worked demonstration** using only
supplied example functions. It does not depend on completing any exercise. We will
create a finite sample from two unbounded generators, normalize accepted readings, and
plot the results. All code is visible below.

The numerical model is deliberately invented for teaching: a baseline, a smooth
oscillation, a slow drift, and a small deterministic ripple. It is not a fitted model of
room physics. Because it uses no clock or random source, asking for the value at the
same minute with the same baseline gives the same result. That makes comparisons
repeatable.

In [347]:
def model_temperature(minute, baseline_c):
    oscillation_c = 1.8 * sin(2.0 * pi * minute / 30.0)
    drift_c = 0.035 * minute
    ripple_c = 0.12 * cos(2.0 * pi * minute / 7.0)
    return baseline_c + oscillation_c + drift_c + ripple_c


def simulated_records(sensor_id, unit, baseline_c):
    """Yield illustrative records; callers must bound this unbounded source."""
    if unit not in ("C", "F"):
        raise ValueError("Use C or F for this simulator.")
    for minute in count():
        value_c = model_temperature(minute, baseline_c)
        value = value_c if unit == "C" else value_c * 9.0 / 5.0 + 32.0
        yield {
            "sensor_id": sensor_id,
            "minute": minute,
            "value": value,
            "unit": unit,
            "status": "ok",
        }

The value model is a pure calculation. The generator adds the source position and yields
records on demand. It does not sleep between readings and does not run in the background.
Here `minute` is simulated time, not a promise about elapsed notebook time. Since this is
a generator function, even its configuration check runs when iteration begins, not when
the generator object is first created.

In [348]:
kitchen_source = islice(simulated_records("kitchen", "C", 21.0), 60)
study_source = islice(simulated_records("study", "F", 22.0), 60)
simulated_batch = list(chain(kitchen_source, study_source))

# Every record in this particular demonstration is deliberately valid.
assert all(validation_error(record) is None for record in simulated_batch)
canonical_demo = list(map(normalise_reading, simulated_batch))
print("Records:", len(canonical_demo))
print("First normalized record:", canonical_demo[0])

Records: 120
First normalized record: {'sensor_id': 'kitchen', 'minute': 0, 'temperature_c': 21.12}


The two bounded sources give us 60 readings per room. `chain` puts one room's batch after
the other; it does not simulate concurrent arrival or reorder messages by time. The
exercise will handle mixed ordering and rejected messages as well.

### Plot the supplied demonstration

The next two figures are display actions outside the calculations. They use no hidden
plotting helper. Matplotlib is the only optional dependency. In an environment without
it, run `%pip install matplotlib` in a new notebook cell, then rerun these plotting cells.
The rest of the notebook does not depend on these plots.

In [349]:
try:
    import matplotlib.pyplot as plt

    get_ipython().run_line_magic("matplotlib", "inline")
    HAVE_MATPLOTLIB = True
except ImportError:
    HAVE_MATPLOTLIB = False
    print("Optional plots skipped. Install matplotlib to show them.")

Optional plots skipped. Install matplotlib to show them.


In [350]:
if HAVE_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for room in ("kitchen", "study"):
        room_records = [
            record for record in canonical_demo if record["sensor_id"] == room
        ]
        minutes = [record["minute"] for record in room_records]
        values = [record["temperature_c"] for record in room_records]
        ax.plot(minutes, values, label=room)
    ax.axhline(23.0, linestyle="--", label="illustrative alert threshold")
    ax.set(
        xlabel="Simulated minute",
        ylabel="Temperature (°C)",
        title="Two sensor formats, one Celsius view",
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

In [351]:
kitchen_values = [
    record["temperature_c"]
    for record in canonical_demo
    if record["sensor_id"] == "kitchen"
]
demo_states = accumulate(kitchen_values, advance_totals, initial=(0, 0.0))
demo_means = [total / count_ for count_, total in demo_states if count_ > 0]

if HAVE_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(kitchen_values, label="individual reading")
    ax.plot(demo_means, linewidth=2.2, label="mean of all readings so far")
    ax.set(
        xlabel="Simulated minute",
        ylabel="Temperature (°C)",
        title="A running mean describes history, not just the latest reading",
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

The running mean reacts slowly after much history has accumulated. That is expected,
not a filtering bug: every earlier reading still contributes. A last-five-readings
average would answer a different question.

### Your integration task

Now combine the ideas, without creating a framework. Keep only the stages we need:
validate and normalize with E11; collect Celsius values by room; summarize each room;
return the report and the rejection information. Treat it as a **finite batch** task.
A complete per-room list is useful here, but it is not a constant-memory solution for an
unbounded feed.

The reported mean is an arithmetic **sample mean**, not automatically a time-weighted
mean. A count of hot samples is not automatically the time spent above the threshold.
Our illustration samples each room at regular intervals, but the function below promises
only a summary of its accepted input values.

<a id="e13"></a>

## Ex13: Build a per-room report with an audit trail

Complete `build_room_report(raw_records, threshold_c=23.0)`. Reuse **E11's
`prepare_batch`** and the supplied `summarise`. Return:

```python
{
    "rooms": {
        "kitchen": {"count": ..., "mean_c": ..., "hot_count": ...},
        # Other rooms that have at least one accepted reading.
    },
    "rejected": [{"index": ..., "reason": ...}, ...],
}
```

Group by `sensor_id`, which is the room label in this classroom example. Preserve all
rejection records. Do not print or plot inside the function. Do not change the raw data.
Empty or wholly invalid input has an empty `rooms` dictionary; do not invent a mean of
zero. A room with no accepted readings must not appear in `rooms`. Assume a finite list
or tuple of input messages and a finite threshold. Room-key order is not graded.

**Hint.** Build a fresh dictionary of room → list of temperatures. `setdefault(room, [])` gives a list for a room, creating it the first time. This local mutation does not alter raw records.

In [352]:
def build_room_report(
    raw_records: list[dict[str, Any]], threshold_c: float = 23.0
) -> dict[str, Any]:
    batch = prepare_batch(raw_records)
    rooms = {}
    for rec in batch["accepted"]:
        room = rec["sensor_id"]
        rooms.setdefault(room, []).append(rec["temperature_c"])
    return {
        "rooms": {
            room: summarise(values, threshold_c) for room, values in rooms.items()
        },
        "rejected": batch["rejected"],
    }

In [353]:
def check_ex13():
    records = [
        {
            "sensor_id": "kitchen",
            "minute": 0,
            "value": 21.0,
            "unit": "C",
            "status": "ok",
        },
        {"sensor_id": "study", "minute": 0, "value": 77.0, "unit": "F", "status": "ok"},
        {
            "sensor_id": "kitchen",
            "minute": 1,
            "value": 23.0,
            "unit": "C",
            "status": "ok",
        },
        {"sensor_id": "study", "minute": 1, "value": 72.5, "unit": "F", "status": "ok"},
        {
            "sensor_id": "garage",
            "minute": 0,
            "value": None,
            "unit": "C",
            "status": "ok",
        },
    ]
    snapshot = deepcopy(records)
    report = build_room_report(records, 23.0)
    assert report["rooms"]["kitchen"] == {"count": 2, "mean_c": 22.0, "hot_count": 1}
    assert report["rooms"]["study"] == {"count": 2, "mean_c": 23.75, "hot_count": 1}
    assert "garage" not in report["rooms"]
    assert report["rejected"] == [{"index": 4, "reason": validation_error(records[4])}]
    assert sum(room["count"] for room in report["rooms"].values()) + len(
        report["rejected"]
    ) == len(records)
    assert build_room_report(records, 22.0)["rooms"]["study"]["hot_count"] == 2
    assert records == snapshot
    assert build_room_report([]) == {"rooms": {}, "rejected": []}
    assert build_room_report([{}])["rooms"] == {}
    assert len(build_room_report([{}])["rejected"]) == 1


run_check("Ex13", check_ex13)

Ex13: all supplied checks passed.


<a id="s14"></a>

# 14. Optional: recursion when the data are nested

> **Extension:** “A home contains floors, floors contain rooms, and rooms may contain sub-zones.”

A function is **recursive** when it calls itself. This is useful when the data contain
smaller structures with the same shape. A zone below contains its own device powers and
a list of child zones. A leaf zone has no children. We assume a finite tree with no
cycles; arbitrary object graphs require additional care.

We will store illustrative nominal device powers in watts. Summing those values is not
an estimate of electrical energy use without device operating times and other assumptions.

In [354]:
home = {
    "name": "home",
    "devices_w": [5.0],
    "children": [
        {"name": "kitchen", "devices_w": [40.0, 60.0], "children": []},
        {
            "name": "study",
            "devices_w": [100.0],
            "children": [{"name": "desk", "devices_w": [8.0], "children": []}],
        },
    ],
}


def count_devices(zone):
    own_count = len(zone["devices_w"])
    child_count = sum(count_devices(child) for child in zone["children"])
    return own_count + child_count


print("Number of devices:", count_devices(home))

Number of devices: 5


There are two things to explain in every recursive solution. **Progress:** the recursive
call receives a child, a smaller part of this finite tree. **Stopping:** a leaf has an
empty child list, whose sum is zero, so no further calls are made.

Recursion is not a defining test of functional programming. A recursive function can
mutate global data, and a loop-based function can expose a pure interface. For flat
sequences in Python, iteration is usually the more straightforward choice. Deep recursive
inputs can exceed the interpreter's recursion limit; increasing that limit is not a
general design solution. See [`sys.getrecursionlimit`](https://docs.python.org/3.13/library/sys.html#sys.getrecursionlimit).

<a id="e14"></a>

## Ex14: Sum device power through a zone tree

Write `total_device_power(zone)` recursively. Each zone has `devices_w`, a list of finite non-negative numbers, and `children`, a list of zones with the same shape. Return the sum of powers in this zone and all descendants. Empty lists contribute zero. Do not modify the tree. Assume a finite, reasonably shallow tree with no cycles.

**Hint.** The answer is the sum of this zone’s powers plus the sum of the answers for its children.

In [355]:
def total_device_power(zone):
    return sum(zone["devices_w"]) + sum(
        total_device_power(child) for child in zone["children"]
    )

In [356]:
def check_ex14():
    tree = {
        "devices_w": [5.0],
        "children": [
            {"devices_w": [10.0, 15.0], "children": []},
            {"devices_w": [100.0], "children": [{"devices_w": [2.0], "children": []}]},
        ],
    }
    snapshot = deepcopy(tree)
    assert isclose(total_device_power(tree), 132.0)
    assert total_device_power({"devices_w": [], "children": []}) == 0
    assert total_device_power({"devices_w": [12.0], "children": []}) == 12.0
    assert tree == snapshot


run_check("Ex14", check_ex14)

Ex14: all supplied checks passed.


<a id="s15"></a>

# 15. Optional: reuse results with memoization

> **Extension:** “The planning screen keeps evaluating the same temperature scenario.”

**Memoization** means remembering results for previously seen inputs so repeated requests
can reuse them. Stable, explicit-input calculations are good candidates. A sensor read,
a clock lookup, or a function that must send a notification is not interchangeable with
a previously stored result.

Python's `lru_cache` creates a caching wrapper. “Least recently used” describes which
stored entries it discards when the chosen size limit is reached. We can apply it as a
higher-order function or with the `@` decorator notation. The following deliberately
cheap example demonstrates cache behaviour, not a claimed speed improvement.

In [357]:
cached_conversion = lru_cache(maxsize=32)(f_to_c)
print(cached_conversion(77.0))  # First request for this key: a miss.
print(cached_conversion(77.0))  # Same key: a hit.
print(cached_conversion(68.0))  # A different key: a miss.
print(cached_conversion.cache_info())

25.0
25.0
20.0
CacheInfo(hits=1, misses=2, maxsize=32, currsize=2)


A **decorator** takes a function and supplies a replacement or wrapper. The notation
below applies the caching wrapper when the function is defined. It is convenient syntax
for a higher-order operation; it is not a separate style of function execution.

The decorator notation is specified in [PEP 318](https://peps.python.org/pep-0318/).

In [358]:
@lru_cache(maxsize=32)
def demo_prediction(initial_c, outside_c, retention, steps):
    current_c = initial_c
    for _ in range(steps):
        current_c = outside_c + retention * (current_c - outside_c)
    return current_c


print(demo_prediction(24.0, 10.0, 0.9, 5))
print(demo_prediction(24.0, 10.0, 0.9, 5))
print(demo_prediction.cache_info())

18.26686
18.26686
CacheInfo(hits=1, misses=1, maxsize=32, currsize=1)


Our illustrative model retains a fraction of the difference from the outside temperature
at each step. It has no feedback from an actual building and is not a heating prediction
validated against measurements. The loop rebinds a local number; all conditions of the
scenario are arguments.

Caching comes with a contract. Arguments must be hashable for `lru_cache` keys; a list
or dictionary is not suitable as an ordinary cache argument. Avoid returning a mutable
object that callers might edit, because a later cache hit can return that same edited
object. Include every result-affecting setting in the inputs. A changing global setting
can otherwise make stored answers stale. A bounded cache also still uses memory.

The wrapper maintains internal cache state, so it is not strictly free of observable
state changes; `cache_info` exposes them. The underlying scalar result can nevertheless
remain determined by its inputs. Near-but-not-equal floating-point inputs do not share a
key automatically, and continuously changing sensor values may offer few useful hits.
Measure before assuming a cache helps. See the
[`lru_cache` documentation](https://docs.python.org/3.13/library/functools.html#functools.lru_cache).

<a id="e15"></a>

## Ex15: Cache an explicit-input scenario

Implement `cached_forecast(initial_c, outside_c, retention, steps)` with
`@lru_cache(maxsize=32, typed=True)`. Start at `initial_c`, then perform exactly `steps` updates of
`current_c = outside_c + retention * (current_c - outside_c)`. Return a number.

`steps` must be a non-negative built-in integer, excluding a Boolean. `retention` must
lie between 0 and 1, inclusive. Raise `ValueError` for either invalid condition. Assume
finite numerical temperatures and a finite numerical retention argument.

For zero steps, return the initial temperature. Keep all model settings explicit. `typed=True` keeps an integer step count distinct
from an equal-valued Boolean or float in the cache, so a hit cannot bypass this type rule.
The exercise intentionally demonstrates result reuse with a small model rather than making
a timing or speed-up claim.

**Hint.** Use the same ordinary loop as the worked model, add validation, and decorate the function. Clear the cache before inspecting hit/miss counts in tests.

In [359]:
@lru_cache(maxsize=32, typed=True)
def cached_forecast(
    initial_c: float, outside_c: float, retention: float, steps: int
) -> float:
    if not isinstance(steps, int) or isinstance(steps, bool) or steps < 0:
        raise ValueError("steps must be a non-negative integer")
    if not (0.0 <= retention <= 1.0):
        raise ValueError("retention must be between 0 and 1, inclusive")

    current_c = initial_c
    for _ in range(steps):
        current_c = outside_c + retention * (current_c - outside_c)
    return current_c

In [360]:
def check_ex15():
    cached_forecast.cache_clear()
    first = cached_forecast(24.0, 10.0, 0.5, 2)
    second = cached_forecast(24.0, 10.0, 0.5, 2)
    assert isclose(first, 13.5)
    assert first == second
    info = cached_forecast.cache_info()
    assert info.misses == 1 and info.hits == 1
    assert info.maxsize == 32
    assert cached_forecast(24.0, 10.0, 0.5, 0) == 24.0
    assert cached_forecast(24.0, 10.0, 1.0, 5) == 24.0
    assert cached_forecast(24.0, 10.0, 0.0, 5) == 10.0
    assert_raises(ValueError, cached_forecast, 24.0, 10.0, 1.1, 2)
    assert_raises(ValueError, cached_forecast, 24.0, 10.0, 0.5, -1)
    assert cached_forecast(24.0, 10.0, 0.5, 1) == 17.0
    assert_raises(ValueError, cached_forecast, 24.0, 10.0, 0.5, 1.0)
    assert_raises(ValueError, cached_forecast, 24.0, 10.0, 0.5, 1.5)
    assert_raises(ValueError, cached_forecast, 24.0, 10.0, 0.5, True)


run_check("Ex15", check_ex15)

Ex15: all supplied checks passed.


<a id="s16"></a>

# 16. Choose the simplest clear design

The story began with three temperatures and ended with a finite-batch report, reusable
policies, a streaming demonstration, and explicit validation. None of those improvements
required a new package or a ban on ordinary Python.

**The main lesson is not “replace every loop.” It is “make calculations, dependencies,
state, and effects easy to see.”** Python supports a mixture of approaches; the design
should help the reader understand the task.

### A decision guide

| Situation | A clear starting point | What to watch |
|:--|:--|:--|
| Transform a short finite sequence | A comprehension or `map` with a named function | Whether a list is actually needed. |
| Select matching records | A comprehension or `filter` with a predicate | Units, boundaries, and whether rejection reasons matter. |
| Compute a standard total or extremum | `sum`, `min`, or `max` | Empty-input policy and numerical meaning. |
| Compute a custom small summary state | A loop with a pure step, or `reduce` | State shape, initialization, and iteration order. |
| Reuse settings with an existing function | `partial` | Mutable referenced settings and keyword overrides. |
| Create a new configured rule | A closure factory | Late binding and changing captured objects. |
| Process an unbounded source | A generator and a bounded consumer | Exhaustion, blocking reads, and accidental materialization. |
| Preserve source data | Fresh outputs; copy changed nested fields | A shallow copy is not a fully independent object graph. |
| Update a device decision | A pure state transition plus an outer action | Calculated intent is not confirmed device success. |
| Repeated expensive deterministic scenario | Consider bounded memoization | Cache keys, stale dependencies, memory, and actual hit rate. |

### Common misunderstandings to leave behind

A lambda is not more functional than a named function. A higher-order function is not
necessarily pure. A loop with local variables can still implement a pure interface.
Returning a new outer container does not make its nested contents immutable. A lazy
iterator carries state and can run out. A running average is not a recent-window average.
A decorator can add side effects rather than remove them. A clever one-liner is not
necessarily easier to test or maintain.

A related Python trap is a mutable default such as `history=[]`: that object is created
once, not freshly for every call. For a helper that needs a new list per call, use `None`
as the default and create the list inside the function. For a functional design, often
the better question is whether history should be an explicit input and returned result.
The [Programming FAQ](https://docs.python.org/3.13/faq/programming.html#why-are-default-values-shared-between-objects)
explains the shared-default behaviour.

### Small glossary

| Term | Meaning in this lesson |
|:--|:--|
| Pure function | A value calculation determined by explicit inputs, without externally observable changes. |
| Side effect | An observable action beyond returning a calculation's result. |
| Referential transparency | The value-based idea that replacing a calculation by its result preserves meaning. |
| Mutation | A change to an existing object. |
| First-class function | A function that can be stored, passed, and returned like other values. |
| Higher-order function | A function that accepts or returns another function. |
| Predicate | A function expressing a true/false condition. |
| Closure | A function with retained access to variables in an enclosing scope. |
| Partial application | Supplying some arguments now to create a callable for the remaining arguments. |
| Composition | Connecting one function's result to another function's input. |
| Reduction | Combining inputs into a final result by repeated steps. |
| Iterable / iterator | Something we can iterate over / a cursor supplying successive items. |
| Generator | An iterator produced by a generator function or expression. |
| Laziness | Computing results when requested rather than eagerly building them all. |
| Memoization | Reusing stored results for repeated inputs. |

### Check your exercise progress

This status display reflects checks run in the current kernel. A pass means the supplied checks passed, not that every possible case has been proved.

In [362]:
for number in range(1, 16):
    label = f"E{number:02d}"
    optional = " (optional)" if number >= 14 else ""
    print(f"{label}{optional}: {exercise_status.get(label, 'not checked')}")

passed = sum(status == "passed" for status in exercise_status.values())
print(f"\nSupplied checks passed: {passed}/15 exercise groups.")

E01: not checked
E02: not checked
E03: not checked
E04: passed
E05: not checked
E06: passed
E07: passed
E08: not checked
E09: not checked
E10: not checked
E11: not checked
E12: not checked
E13: not checked
E14 (optional): not checked
E15 (optional): not checked

Supplied checks passed: 15/15 exercise groups.


### Sources and further reading

The smart-home narrative, data, exercise requirements, and worked code are original
teaching examples. The links below document the Python behaviour used in them. Python
3.13 documentation is used as a stable reference; the notebook code uses features
available from Python 3.10 onwards.

| Read about | Primary reference |
|:--|:--|
| Functional style in a multi-paradigm language | [Python Functional Programming HOWTO](https://docs.python.org/3.13/howto/functional.html) |
| Function objects, parameters, and lambdas | [Python tutorial: More Control Flow Tools](https://docs.python.org/3.13/tutorial/controlflow.html) |
| Comprehensions and collections | [Python tutorial: Data Structures](https://docs.python.org/3.13/tutorial/datastructures.html) |
| `sorted`, `map`, `filter`, `any`, and `all` | [Built-in Functions](https://docs.python.org/3.13/library/functions.html) |
| Immutable and mutable sequence types | [Built-in Types](https://docs.python.org/3.13/library/stdtypes.html) |
| Shallow and deep copying | [The copy module](https://docs.python.org/3.13/library/copy.html) |
| Closure late binding and mutable defaults | [Programming FAQ](https://docs.python.org/3.13/faq/programming.html) |
| `partial`, `reduce`, and `lru_cache` | [The functools module](https://docs.python.org/3.13/library/functools.html) |
| Partial application versus currying | [PEP 309](https://peps.python.org/pep-0309/) |
| Decorator syntax and meaning | [PEP 318](https://peps.python.org/pep-0318/) |
| Generator execution | [Python tutorial: Generators](https://docs.python.org/3.13/tutorial/classes.html#generators) |
| Generator-expression evaluation | [Python Language Reference: Generator expressions](https://docs.python.org/3.13/reference/expressions.html#generator-expressions) |
| Iterator building blocks | [The itertools module](https://docs.python.org/3.13/library/itertools.html) |
| Finite-number checks | [math.isfinite](https://docs.python.org/3.13/library/math.html#math.isfinite) |
| Assertions and optimized execution | [The assert statement](https://docs.python.org/3.13/reference/simple_stmts.html#the-assert-statement) |
| Recursion limit | [sys.getrecursionlimit](https://docs.python.org/3.13/library/sys.html#sys.getrecursionlimit) |
| Background on two-threshold hysteresis | [MathWorks: Relay block](https://www.mathworks.com/help/simulink/slref/relay.html) — a generic relay example; the heating rule in E12 has its own explicitly stated switching direction. |

The examples are intentionally small and educational - they may help you to increase your app design capabilities.